In [10]:
import time
from PIL import ImageGrab
import pyautogui
import numpy as np

# PyAutoGUI 安全機制
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0

# ==================== 設定參數 ====================
# 【1. X 軸動態偵測範圍】
# 低速時的起點與寬度
BASE_SCAN_LEFT = 110
BASE_SCAN_RIGHT = 150

# 高速時的最大延伸距離
MAX_SCAN_LEFT = 180
MAX_SCAN_RIGHT = 240

# 【2. Y 軸高度分區】
HIGH_BIRD_TOP = 360      # 高空鳥頂端
GROUND_TOP = 415         # 地面障礙頂端
GROUND_BOTTOM = 470      # 地面障礙底端

# 仙人掌與翼龍目標顏色
COLOR_DAY = (83, 83, 83)       # 白天深灰
COLOR_NIGHT = (172, 172, 172)  # 夜間淺灰
TOLERANCE = 5

# 【時間與急降參數】
BASE_JUMP_WAIT = 0.28   # 初始起跳下壓等待時間
BASE_COOLDOWN = 0.001   # 跳躍冷卻時間
DROP_DURATION = 0.05    # 按住向下鍵時間

MIN_JUMP_WAIT = 0.08
MIN_COOLDOWN = 0.05
# ==================================================

def is_night_mode(arr):
    """透過檢查區域內的平均亮度判斷當前是白天還是夜間模式"""
    return np.mean(arr) < 120

def check_color_pixels(arr, tr, tg, tb):
    """計算符合目標顏色的像素數量"""
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    mask = (
        (np.abs(r - tr) <= TOLERANCE) &
        (np.abs(g - tg) <= TOLERANCE) &
        (np.abs(b - tb) <= TOLERANCE)
    )
    return np.count_nonzero(mask)

def detect_obstacles(cropped_arr):
    """
    從裁剪好的小圖判斷是否有地面障礙物或高空鳥
    """
    tr, tg, tb = COLOR_NIGHT if is_night_mode(cropped_arr) else COLOR_DAY

    high_crop_height = GROUND_TOP - HIGH_BIRD_TOP
    high_crop = cropped_arr[:high_crop_height, :]
    ground_crop = cropped_arr[high_crop_height:, :]

    ground_count = check_color_pixels(ground_crop, tr, tg, tb)
    high_count = check_color_pixels(high_crop, tr, tg, tb)

    has_ground = ground_count > 3
    has_high_bird = high_count > 3

    return has_ground, has_high_bird

def quick_jump(wait_time):
    """執行跳躍並快速落地"""
    pyautogui.press('space')
    time.sleep(wait_time)
    pyautogui.keyDown('down')
    time.sleep(DROP_DURATION)
    pyautogui.keyUp('down')

def duck_down():
    """遇到高空鳥時執行蹲下避開"""
    pyautogui.keyDown('down')
    time.sleep(0.25)
    pyautogui.keyUp('down')

def main():
    print("=" * 40)
    print("Chrome 小恐龍 - 自適應速度 (動態 X 軸距) 版")
    print("=" * 40)
    print(f"低速偵測 X 軸：{BASE_SCAN_LEFT} ~ {BASE_SCAN_RIGHT}")
    print(f"高速偵測 X 軸：預計推進至 {MAX_SCAN_LEFT} ~ {MAX_SCAN_RIGHT}")
    print("=" * 40)

    for i in range(5, 0, -1):
        print(f"倒數 {i} 秒後開始...")
        time.sleep(1)

    print("\n[開始執行] 不斷監視畫面中...")

    last_jump = 0
    was_detected = False
    game_start_time = None

    try:
        while True:
            now = time.time()

            # 根據遊戲時間動態計算當前的 X 軸範圍與等待時間
            if game_start_time is not None:
                elapsed = now - game_start_time
                
                # 每秒增加 X 軸距離（前 35 秒內從近處平滑推到遠處）
                x_offset = min(70, int(elapsed * 2.0))
                scan_left = BASE_SCAN_LEFT + x_offset
                scan_right = BASE_SCAN_RIGHT + x_offset

                # 急降等待時間與冷卻時間動態遞減
                speed_factor = max(0.0, elapsed * 0.0025)
                current_wait = max(MIN_JUMP_WAIT, BASE_JUMP_WAIT - speed_factor)
                current_cooldown = max(MIN_COOLDOWN, BASE_COOLDOWN - speed_factor)
            else:
                scan_left = BASE_SCAN_LEFT
                scan_right = BASE_SCAN_RIGHT
                current_wait = BASE_JUMP_WAIT
                current_cooldown = BASE_COOLDOWN

            if now - last_jump < current_cooldown:
                time.sleep(0.001)
                continue

            # 5 秒沒跳躍自動重置（判定為死亡重來）
            if game_start_time and (now - last_jump > 5.0):
                game_start_time = None

            # 動態範圍畫面擷取
            img = ImageGrab.grab(bbox=(scan_left, HIGH_BIRD_TOP, scan_right, GROUND_BOTTOM))
            arr = np.array(img)

            has_ground, has_high_bird = detect_obstacles(arr)

            # 情況 A：地面仙人掌/低空鳥 -> 跳躍
            if has_ground and not was_detected:
                if game_start_time is None:
                    game_start_time = now

                quick_jump(current_wait)
                last_jump = time.time()
                was_detected = True

            # 情況 B：高空鳥 -> 蹲下
            elif has_high_bird and not has_ground and not was_detected:
                if game_start_time is None:
                    game_start_time = now

                duck_down()
                last_jump = time.time()
                was_detected = True

            # 區域清空重置狀態
            if not has_ground and not has_high_bird:
                was_detected = False

            time.sleep(0.001)

    except pyautogui.FailSafeException:
        print("\n[停止] 觸發 Failsafe（滑鼠已移至螢幕左上角）")
    except KeyboardInterrupt:
        print("\n[停止] 手動中斷")
    except Exception as e:
        print(f"\n[錯誤] {e}")

if __name__ == "__main__":
    main()

Chrome 小恐龍 - 自適應速度 (動態 X 軸距) 版
低速偵測 X 軸：110 ~ 150
高速偵測 X 軸：預計推進至 180 ~ 240
倒數 5 秒後開始...
倒數 4 秒後開始...
倒數 3 秒後開始...
倒數 2 秒後開始...
倒數 1 秒後開始...

[開始執行] 不斷監視畫面中...

[停止] 手動中斷
